# Retrieval-Augmented Generation (RAG) Pipeline

This notebook demonstrates the complete RAG pipeline.

The pipeline is divided into two major phases:

1. **Document Ingestion and Indexing**
   - Read the document
   - Preprocess the extracted text
   - Create chunks
   - Generate embeddings
   - Store the embeddings in a vector store

2. **Query and Response Generation**
   - Receive a user query
   - Generate the query embedding
   - Retrieve relevant document chunks
   - Build a prompt using the query and retrieved context
   - Generate the final response using an LLM

# Phase 1: Document Ingestion and Indexing

In this phase, the source document is transformed into searchable vector representations.

The document passes through the following pipeline:

PDF
↓
Text Extraction
↓
Text Preprocessing
↓
Chunking
↓
Embedding Generation
↓
Vector Store

## 1. PDF Reading and Text Extraction

The first step is to extract the content from the PDF document.

The document is initially converted into a page-wise representation.

Each element of the resulting list represents the extracted text from one page of the PDF.

This page-wise representation is preserved because the page boundary provides useful structural information for subsequent preprocessing.

In [1]:
from pypdf import PdfReader

In [2]:
pdf = []
try:
    reader = PdfReader('./src/pdfs/bom.pdf')
    for page in reader.pages:
         pdf.append(page.extract_text())

    print(f"Total Pages: {len(pdf)}")

except Exception as exc:
    # TO Catch error 
    print("Remaining")

Total Pages: 224


## 2. Text Preprocessing

The raw text extracted from a PDF may contain structural noise such as:

- Repeated headers
- Repeated footers
- Page numbers
- Other boilerplate content

The purpose of this stage is to identify and remove repeated structural content while preserving the actual semantic content of the document.

The preprocessing pipeline uses similarity analysis and statistical patterns to identify boilerplate content dynamically rather than relying on document-specific hardcoded strings.

In [3]:
from src.preprocessing.cleaner import normalize_pdf
from src.preprocessing.similarity import compute_line_similarity
from src.preprocessing.boilerplate import identify_boilerplate, remove_boilerplate

In [4]:
# Phase 1: PDF Text normalization
normalized_pdf = normalize_pdf(pdf)

# # Phase 2: Statistical Analysis
similar_line_analysis = compute_line_similarity(pdf=normalized_pdf, candidate_lines=3)

# Phase 3: Boilerplate Identification
candidate_boilerplates = identify_boilerplate(similarity_df=similar_line_analysis)

# # Phase 4: Boilerplate Removal
boilerplate_config = (
    candidate_boilerplates[
        candidate_boilerplates["is_boilerplate"]
    ][["region", "line"]]
)



filtered_pdf = remove_boilerplate(normalized_pdf, boilerplate_config=boilerplate_config)

# Phase 5: Converting page-wise PDF into the one combined text
text = "".join(filtered_pdf)

---------------------------------------
------------- Original PDF ------------
Basics of Mathematics 
Sr. No. Topic name Page number 
1. Qualitative and Quantitative Analysis 2 
2. Number and Number System, problems on Numbers 7 
3. Fractions, Decimals and Numbers 32 
4. Divisibility and Divisibility Test 39 
5. HCF and LCM 44 
6. Squares, Cubes, Square-roots & Cube-roots 51 
7. Simplifications 58 
8. Ratio and Proportion 69 
9. Chain rule 
74 
10. Averages 82 
11. Percentages 91 
12. Profit and Loss 101 
13. Simple and Compound Interest 110 
14. Heights, Distances & Basic Trigonometry 135 
15. Time, Work and Wages 142 
16. Time and Distances 162 
17. Trains, Boats and Streams 176 
18. Clock 197 
19. Series 205 
20. Problems on Ages   210 
21. Discounts – its types and Analysis 216 
 
 
 
---------------------------------------

---------------------------------------
------------- Cleaned PDF ------------
Basics of Mathematics 
Sr. No. Topic name Page number 
1. Qualitative and Quant

## 3. Text Chunking

The cleaned document is divided into smaller text segments called chunks.

Chunking is necessary because the entire document cannot be efficiently embedded and retrieved as a single unit.

The objective of chunking is to create meaningful units of text that:

- Preserve sufficient semantic context
- Are small enough for efficient embedding and retrieval
- Can be independently retrieved in response to a user query

In [5]:
from src.chunking.chunker import text_splitter

In [6]:
chunks = text_splitter(text=text)


for i, chunk in enumerate(chunks):
    print(f"\n--- Chunk {i+1} ---")
    print(f"\n--- First half of chunk {i+1} ---")
    print(chunk[:300])
    print(f"\n--- Second half of chunk {i+1} ---")
    print(chunk[300:])
    print(f"\nLength: {len(chunk)}")

    if i == 2:
        break

Split PDF text into 22 sub-documents.

--- Chunk 1 ---

--- First half of chunk 1 ---
Sr. No. Topic name Page number
1. Qualitative and Quantitative Analysis 2
2. Number and Number System, problems on Numbers 7
3. Fractions, Decimals and Numbers 32
4. Divisibility and Divisibility Test 39
5. HCF and LCM 44
6. Squares, Cubes, Square-roots & Cube-roots 51
7. Simplifications 58
8. Ratio

--- Second half of chunk 1 ---
 and Proportion 69
9. Chain rule
74
10. Averages 82
11. Percentages 91
12. Profit and Loss 101
13. Simple and Compound Interest 110
14. Heights, Distances & Basic Trigonometry 135
15. Time, Work and Wages 142
16. Time and Distances 162
17. Trains, Boats and Streams 176
18. Clock 197
19. Series 205
20. Problems on Ages 210
21. Discounts – its types and Analysis 216Section A
A.1 Introduction to Qualitative and
Quantitative Analysis
Qualitative Data Analysis
Qualitative data analysis (QDA) is the process of organizing, analyzing, and interpreting
qualitative data—non-numeric, c

## 4. Document Embedding Generation

Each text chunk is converted into a numerical vector representation called an embedding.

The embedding represents the semantic meaning of the chunk in a vector space.

Semantically similar chunks should have similar vector representations.

The result of this stage is:

Text Chunks
        ->
Embedding Model
        ->
Vector Representations

In [7]:
from src.embeddings.embedder import chunk_encoder

In [8]:
embeddings = chunk_encoder(chunks)

embeddings

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Each embedding has 384 dimensions
Similarity matrix:
[[ 1.          0.60902035  0.46741772  0.41765928  0.31461573  0.39369875
   0.34920907  0.48842028  0.559707    0.50960577  0.61037403  0.46877468
   0.2865466   0.28245527  0.25410283  0.30219734  0.2787121   0.2485761
   0.22322056  0.3150742   0.3674897   0.30143046]
 [ 0.60902035  0.9999999   0.8082719   0.70771253  0.5715337   0.6755384
   0.6358664   0.75199354  0.54995906  0.6357827   0.66227096  0.29256487
   0.11715031  0.10368641  0.04718842  0.0780332   0.1097618   0.07927002
   0.07696318  0.13564862  0.14791796  0.13884303]
 [ 0.46741772  0.8082719   1.          0.852163    0.59900427  0.65352976
   0.584928    0.70955765  0.52519727  0.63953185  0.63214946  0.26657555
   0.10993231  0.10303891  0.04962326  0.08211279  0.10259019  0.06191434
   0.07400539  0.10947983  0.1051769   0.13139331]
 [ 0.41765928  0.70771253  0.852163    1.          0.7548932   0.7117351
   0.6359838   0.6850221   0.45739776  0.56506455  0.5780

array([[ 0.04711536,  0.03406747, -0.04829038, ..., -0.03649592,
         0.00332668, -0.00989808],
       [ 0.07780587,  0.01126292, -0.0926613 , ...,  0.03075932,
         0.01594057, -0.0830707 ],
       [ 0.07754667,  0.03618181, -0.09084365, ..., -0.01225633,
         0.03249075, -0.11427964],
       ...,
       [-0.02720022,  0.01416135, -0.0383908 , ...,  0.02995102,
         0.02452983, -0.03920145],
       [-0.05372377,  0.07342223, -0.02710336, ...,  0.02013069,
         0.00432806, -0.02857885],
       [-0.0286057 ,  0.07702753, -0.05043437, ..., -0.01324628,
        -0.01006904, -0.06025593]], shape=(22, 384), dtype=float32)

## 5. Vector Store

The generated embeddings are stored in a vector index.

The vector store is responsible for:

- Storing document embeddings
- Maintaining the relationship between vectors and their original text chunks
- Searching for vectors that are most similar to a query embedding

In this project, FAISS is used as the vector search backend.

The document ingestion pipeline is now complete:

PDF
->
Text Extraction
->
Preprocessing
->
Chunking
->
Embedding
->
Vector Store

In [11]:
from src.vector_store.vector_store import FaissVectorStore

In [12]:
# Configure FaissVectorStore
dimension = embeddings.shape[1]

vectorStore = FaissVectorStore(dimension=dimension)

In [13]:
# Adding Embeddings Into the Vector Store
vectorStore.add(embeddings=embeddings)

In [14]:
# Checking total Indexes
vectorStore.total_vectors()

22

# Phase 2: Query and Response Generation

The second phase begins when a user submits a question.

The query is transformed into the same vector space as the document chunks so that semantically relevant information can be retrieved.

The pipeline is:

User Query
->
Query Embedding
->
Vector Search
->
Relevant Context
->
Prompt Construction
->
LLM Response Generation

## 1. User Query

The user provides a natural-language question.

The query represents the information that the user wants to retrieve from the indexed document.

In [16]:
# query = "What is qualitative data analysis?"
query = "Steps to effective quantitative data analysis "


## 2. Query Embedding

The user query is converted into an embedding using the same embedding model used for the document chunks.

Using the same embedding model is important because both the query and the document chunks must exist in the same vector space for meaningful similarity comparison.

In [ ]:
from src.embeddings.embedder import chunk_encoder

In [19]:
query_embedding = chunk_encoder([query])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Each embedding has 384 dimensions
Similarity matrix:
[[1.0000001]]


## 3. Similarity Search

The query embedding is passed to the vector store.

The vector store compares the query vector with the stored document vectors and retrieves the most semantically similar chunks.

The result of this stage is the relevant context required to answer the user's question.

In [20]:
# Querying the Vector store
distance, indices = vectorStore.search(query_embedding, 3)

In [21]:
# Preparing the Retrieved text List
retrieved_text = []
for idx in indices[0]:
    retrieved_text.append(chunks[idx])

retrieved_text

['experiences.\n4. Enhanced User Satisfaction: Pinpoints strengths and weaknesses, enabling improvements that\nboost conversions and delight users.\nSteps to effective quantitative data analysis\nQuantitative Data Analysis Steps:\n1. Collect Data: Use surveys, website analytics, heatmaps, or session recordings, focusing on\nclose-ended questions for numerical data.\n2. Clean Data: Remove errors, duplicates, omissions, and outliers to ensure accuracy and\nconsistency.\n3. Analyze & Interpret: Use descriptive analysis (e.g., mean, median) to summarize data or\ninferential analysis (e.g., t-tests) to find relationships and make predictions. Identify actionable\ninsights.\n4. Visualize & Share: Present findings with charts, graphs, or dashboards to highlight patterns\nand relationships.\nDifference between Quantitative data analysis and Qualitative data analysisSection BDivisibility Rule of 1\nDivisibility Rules',
 'Quantitative data analysis is the process of analyzing and interpreting nu

## 4. Prompt Construction and Response Generation

The retrieved document context and the original user query are combined to construct a prompt for the Large Language Model.

The LLM uses the retrieved context to generate the final response.

The final stage is:

User Query
        +
Retrieved Context
        ->
Prompt
        ->
Large Language Model
        ->
Generated Response

In [22]:
from src.generation.llm import LLMResponseGenerator

In [23]:
# LLM Configuration
llm =  LLMResponseGenerator()

In [24]:
# Prompt Building
prompt = llm.prompt_builder(context=retrieved_text, query=query)

In [26]:
# Response Generation
response = llm.generate_response(prompt=prompt)

Quantitative Data Analysis Steps:
1.  **Collect Data:** Use surveys, website analytics, heatmaps, or session recordings, focusing on close-ended questions for numerical data.
2.  **Clean Data:** Remove errors, duplicates, omissions, and outliers to ensure accuracy and consistency.
3.  **Analyze & Interpret:** Use descriptive analysis (e.g., mean, median) to summarize data or inferential analysis (e.g., t-tests) to find relationships and make predictions. Identify actionable insights.
4.  **Visualize & Share:** Present findings with charts, graphs, or dashboards to highlight patterns and relationships.


In [27]:
print(response.text)

Quantitative Data Analysis Steps:
1.  **Collect Data:** Use surveys, website analytics, heatmaps, or session recordings, focusing on close-ended questions for numerical data.
2.  **Clean Data:** Remove errors, duplicates, omissions, and outliers to ensure accuracy and consistency.
3.  **Analyze & Interpret:** Use descriptive analysis (e.g., mean, median) to summarize data or inferential analysis (e.g., t-tests) to find relationships and make predictions. Identify actionable insights.
4.  **Visualize & Share:** Present findings with charts, graphs, or dashboards to highlight patterns and relationships.
